# LeetCode 622 — Design Circular Queue bằng Python

Link đề: https://leetcode.com/problems/design-circular-queue/ · Độ khó: **Medium** · Chủ đề: Design, Queue, Array, Linked List

Skeleton LeetCode:

```python
class MyCircularQueue:

    def __init__(self, k: int):
        ...

    def enQueue(self, value: int) -> bool:
        ...

    def deQueue(self) -> bool:
        ...

    def Front(self) -> int:
        ...

    def Rear(self) -> int:
        ...

    def isEmpty(self) -> bool:
        ...

    def isFull(self) -> bool:
        ...


# Your MyCircularQueue object will be instantiated and called as such:
# obj = MyCircularQueue(k)
# param_1 = obj.enQueue(value)
# param_2 = obj.deQueue()
# param_3 = obj.Front()
# param_4 = obj.Rear()
# param_5 = obj.isEmpty()
# param_6 = obj.isFull()
```

Ghi chú: đây là bài **thiết kế lớp**. LeetCode không gọi một hàm duy nhất mà chạy một **chuỗi lệnh** (tên phương thức + tham số). Notebook tự dựng bộ chạy `run_ops` mô phỏng cách đó.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tự cài đặt một **hàng đợi vòng (circular queue / ring buffer)** có **sức chứa cố định `k`**, với 6 thao tác `enQueue`, `deQueue`, `Front`, `Rear`, `isEmpty`, `isFull`, tất cả nên chạy `O(1)`.
2. Nắm các công cụ Python: lớp (`class`, `__init__`, `self`), list cố định `[0] * k`, toán tử `%` để "quay vòng" chỉ số, lớp `Node` cho danh sách liên kết, và cách mô phỏng LeetCode chạy chuỗi lệnh bằng `getattr`.
3. Đi từ **list thường với `pop(0)` (dequeue `O(n)`)** → **mảng + `front` + `size`** → **mảng `k + 1` ô với hai con trỏ `head`/`tail` (chừa một ô trống)** → **hai bộ đếm tăng mãi** → **danh sách liên kết có đầu/đuôi**.
4. Hiểu mấu chốt: vòng tròn được tạo ra bằng phép **`% k`**, và khó khăn thật sự của bài là **phân biệt hàng đợi rỗng với hàng đợi đầy** khi hai con trỏ trùng nhau.
5. Tránh các lỗi kinh điển: dùng `front + size - 1` mà quên `% k`, quên cập nhật `size`, tính vị trí `Rear` sai khi đã quay vòng, nhầm "rỗng" với "đầy" ở hai con trỏ, và trả `False`/`-1` sai chỗ.

## 2. Đề bài & ví dụ

Thiết kế **hàng đợi vòng** (circular queue). Đây là cấu trúc **FIFO** (First In First Out: vào trước ra trước) trong đó vị trí cuối **nối vòng về** vị trí đầu, tạo thành một vòng tròn, còn gọi là **Ring Buffer**.

Lợi ích: với hàng đợi thường dùng mảng, khi mảng đầy thì không thêm được nữa dù phía trước còn ô trống (vì phần tử đã lấy ra). Hàng đợi vòng **tái sử dụng** các ô trống phía trước.

Cài đặt lớp `MyCircularQueue`:

- `MyCircularQueue(k)`: khởi tạo hàng đợi với sức chứa `k`.
- `Front()`: lấy phần tử **đầu** hàng; rỗng thì trả `-1`.
- `Rear()`: lấy phần tử **cuối** hàng; rỗng thì trả `-1`.
- `enQueue(value)`: thêm `value` vào cuối; trả `True` nếu thành công (không đầy), ngược lại `False`.
- `deQueue()`: xóa phần tử đầu; trả `True` nếu thành công (không rỗng), ngược lại `False`.
- `isEmpty()`: hàng đợi có rỗng không.
- `isFull()`: hàng đợi có đầy không.

Yêu cầu: **không** dùng cấu trúc hàng đợi dựng sẵn của ngôn ngữ.

**Ví dụ 1**

```
Input
["MyCircularQueue", "enQueue", "enQueue", "enQueue", "enQueue", "Rear", "isFull", "deQueue", "enQueue", "Rear"]
[[3], [1], [2], [3], [4], [], [], [], [4], []]
Output
[null, true, true, true, false, 3, true, true, true, 4]
```

Giải thích: tạo hàng đợi sức chứa 3; thêm `1, 2, 3` đều thành công; thêm `4` thất bại vì đã đầy (`False`); `Rear()` là `3`; `isFull()` là `True`; `deQueue()` lấy `1` ra (`True`); `enQueue(4)` giờ thành công; `Rear()` là `4`.

**Ràng buộc:**

- `1 <= k <= 1000`
- `0 <= value <= 1000`
- Tối đa `3000` lần gọi các thao tác.

Chú ý: vì `value >= 0` nên `-1` luôn là giá trị "báo rỗng" không lẫn với dữ liệu thật.

In [1]:
from typing import List, Optional
from collections import deque
import random
import time

def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế: phần tử đầu là tên lớp (hàm khởi tạo),
    các phần tử sau là tên phương thức. Trả về list kết quả (None cho hàm khởi tạo)."""
    obj, out = None, []
    for i, (name, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))
    return out

ex_ops = ["MyCircularQueue", "enQueue", "enQueue", "enQueue", "enQueue", "Rear", "isFull", "deQueue", "enQueue", "Rear"]
ex_args = [[3], [1], [2], [3], [4], [], [], [], [4], []]
ex_expected = [None, True, True, True, False, 3, True, True, True, 4]
print("chuỗi lệnh ví dụ 1 gồm", len(ex_ops), "lệnh")
print("kỳ vọng:", ex_expected)

chuỗi lệnh ví dụ 1 gồm 10 lệnh
kỳ vọng: [None, True, True, True, False, 3, True, True, True, 4]


## 3. Kiến thức nền

### 3.1. Hàng đợi FIFO và cách LeetCode kiểm tra bài thiết kế

**Hàng đợi (queue)** giống xếp hàng mua vé: ai đến trước được phục vụ trước. Thêm vào **cuối** (enqueue), lấy ra từ **đầu** (dequeue).

Với bài "design", LeetCode không truyền một input rồi so một output. Nó tạo một đối tượng, **gọi lần lượt một chuỗi phương thức**, và so sánh danh sách kết quả từng lệnh. Ta đã dựng `run_ops` để làm đúng như vậy. Trong `run_ops`, `getattr(obj, "enQueue")` lấy phương thức theo **tên chuỗi**, rồi `(...)` để gọi.

In [2]:
class Counter3:
    def __init__(self):
        self.n = 0
    def inc(self, d):
        self.n += d
        return self.n

c = Counter3()
name = "inc"
method = getattr(c, name)      # tương đương c.inc
print(method(5), method(2))    # 5 7
print(run_ops(Counter3, ["Counter3", "inc", "inc"], [[], [4], [1]]))

5 7
[None, 4, 5]


### 3.2. Lớp, `__init__` và `self`

`class` gom **dữ liệu** (thuộc tính) và **hành vi** (phương thức) vào một khuôn. `__init__` chạy khi tạo đối tượng và thường dùng để khởi tạo thuộc tính. `self` là chính đối tượng đang được thao tác: `self.size = 0` nghĩa là "đối tượng này có thuộc tính `size` bằng 0". Hai đối tượng khác nhau có `self` khác nhau, nên dữ liệu **không dùng chung**.

In [3]:
class Box:
    def __init__(self, cap):
        self.cap = cap          # thuộc tính
        self.items = []
    def add(self, x):
        if len(self.items) >= self.cap:
            return False
        self.items.append(x)
        return True

a, b = Box(1), Box(2)
print(a.add("x"), a.add("y"))   # True False
print(b.add("x"), b.add("y"), b.add("z"))   # True True False
print(a.items, b.items)

True False
True True False
['x'] ['x', 'y']


### 3.3. List cố định `[0] * k` và gán theo chỉ số

`[0] * k` tạo list `k` phần tử `0`. Ta dùng nó như một **mảng cố định**: không bao giờ `append`/`pop`, chỉ đọc và ghi `q[i]`. Một ô "trống" chỉ có nghĩa theo logic của ta (do `size` và `front` quyết định ô nào đang được dùng), còn trong bộ nhớ nó vẫn chứa giá trị cũ.

In [4]:
q = [0] * 5
q[2] = 9
q[4] = 7
print(q)
q[2] = 3             # ghi đè
print(q, "| độ dài:", len(q))

[0, 0, 9, 0, 7]
[0, 0, 3, 0, 7] | độ dài: 5


### 3.4. Phép `%` và chỉ số "quay vòng"

`a % k` cho **số dư** khi chia `a` cho `k`, luôn nằm trong `0 .. k - 1`. Điều này biến một số đếm tăng mãi thành chỉ số quay vòng:

`0, 1, 2, 3, 4, 5, 6, ...  %  3  →  0, 1, 2, 0, 1, 2, 0, ...`

Đây chính là "vòng tròn": sau ô cuối (`k - 1`) lại về ô `0`. Công thức hay dùng: **vị trí kế tiếp của `i` là `(i + 1) % k`**.

In [5]:
k = 3
print([i % k for i in range(10)])
i = 2
print("sau ô", i, "là ô", (i + 1) % k)   # quay về 0

# Python: % với số âm vẫn không âm. Ngôn ngữ khác (C++/Java) thì có thể âm!
print((-1) % 3)     # 2: lùi một ô từ ô 0 sẽ về ô k-1

[0, 1, 2, 0, 1, 2, 0, 1, 2, 0]
sau ô 2 là ô 0
2


### 3.5. Lớp `Node` và danh sách liên kết

**Danh sách liên kết đơn** gồm các nút; mỗi nút giữ giá trị `val` và tham chiếu `next` đến nút kế. Ta giữ hai con trỏ `head` (nút đầu) và `tail` (nút cuối). Thêm cuối: `tail.next = nút_mới`; xóa đầu: `head = head.next`. Cả hai đều `O(1)`.

In [6]:
class Node:
    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt

head = tail = Node(10)
tail.next = Node(20); tail = tail.next
tail.next = Node(30); tail = tail.next

cur, vals = head, []
while cur:
    vals.append(cur.val)
    cur = cur.next
print(vals)                      # [10, 20, 30]
head = head.next                 # xóa đầu
print(head.val, tail.val)        # 20 30

[10, 20, 30]
20 30


### 3.6. Toán tử `list.pop(0)` rất đắt

`lst.pop(0)` xóa phần tử đầu, nhưng **mọi phần tử còn lại phải dịch sang trái một ô**: `O(n)`. Còn `lst.append(x)` ở cuối là `O(1)`. Vì thế dùng list thường làm hàng đợi thì dequeue chậm. Hãy đo:

In [7]:
for n in (10_000, 40_000, 160_000):
    a = list(range(n))
    t0 = time.perf_counter()
    for _ in range(2000):
        a.pop(0)
    dt = time.perf_counter() - t0
    print(f"n = {n:7d}: 2000 lần pop(0) mất {dt * 1000:7.2f} ms")
print("n càng lớn, mỗi lần pop(0) càng chậm -> O(n)")

n =   10000: 2000 lần pop(0) mất    3.34 ms
n =   40000: 2000 lần pop(0) mất   14.48 ms
n =  160000: 2000 lần pop(0) mất   61.01 ms
n càng lớn, mỗi lần pop(0) càng chậm -> O(n)


## 4. Ý tưởng nền tảng / trực giác

Hãy tưởng tượng một **mặt đồng hồ** có `k` ô (`k = 3` ở ví dụ). Ta đặt phần tử vào các ô, và có một **kim đầu hàng** chỉ vào phần tử cũ nhất. Phần tử mới được đặt vào **ô ngay sau phần tử cuối**; khi lấy ra, kim đầu hàng nhích sang ô kế tiếp. Hết ô `k - 1` thì quay về ô `0`.

Tính tay ví dụ 1 (`k = 3`), ghi nội dung 3 ô `[ô0, ô1, ô2]` và "hàng đợi theo thứ tự":

| Lệnh | 3 ô | Hàng đợi (đầu → cuối) | Kết quả |
|---|---|---|---|
| (tạo) | `[_, _, _]` | `[]` | |
| `enQueue(1)` | `[1, _, _]` | `[1]` | `True` |
| `enQueue(2)` | `[1, 2, _]` | `[1, 2]` | `True` |
| `enQueue(3)` | `[1, 2, 3]` | `[1, 2, 3]` | `True` |
| `enQueue(4)` | `[1, 2, 3]` | `[1, 2, 3]` | `False` (đầy) |
| `Rear()` | | | `3` |
| `isFull()` | | | `True` |
| `deQueue()` | `[_, 2, 3]` | `[2, 3]` | `True` |
| `enQueue(4)` | `[4, 2, 3]` | `[2, 3, 4]` | `True` (đặt vào ô 0 đã trống) |
| `Rear()` | | | `4` |

Điểm thú vị: sau bước `enQueue(4)` cuối, giá trị `4` nằm ở **ô 0** (đầu mảng) nhưng về mặt hàng đợi nó là phần tử **cuối**. Vòng tròn quay lại đầu mảng.

Ta cần theo dõi vừa đủ thông tin để biết: **đầu hàng ở ô nào** và **có bao nhiêu phần tử**. Từ hai thông tin đó, ô cuối hàng là `(đầu + số_phần_tử - 1) % k` và ô để thêm mới là `(đầu + số_phần_tử) % k`.

In [8]:
k = 3
front, size = 0, 0
q = [0] * k
for v in (1, 2, 3):
    pos = (front + size) % k
    q[pos] = v
    size += 1
    print(f"enQueue({v}): ghi vào ô {pos} -> {q}")
front = (front + 1) % k; size -= 1          # deQueue
print("deQueue: front =", front, "size =", size, "| nội dung ô:", q)
pos = (front + size) % k                     # ô để thêm mới
print("ô thêm mới:", pos)
q[pos] = 4; size += 1
print("sau enQueue(4):", q, "| Rear ở ô", (front + size - 1) % k, "=", q[(front + size - 1) % k])

enQueue(1): ghi vào ô 0 -> [1, 0, 0]
enQueue(2): ghi vào ô 1 -> [1, 2, 0]
enQueue(3): ghi vào ô 2 -> [1, 2, 3]
deQueue: front = 1 size = 2 | nội dung ô: [1, 2, 3]
ô thêm mới: 0
sau enQueue(4): [4, 2, 3] | Rear ở ô 0 = 4


## 5. Cách 1: Brute Force / baseline (list thường, `pop(0)`)

**Ý tưởng:** dùng một list Python thông thường `self.a` đóng vai hàng đợi. `enQueue` thì `append` (nếu chưa đầy); `deQueue` thì `pop(0)`; `Front` là `a[0]`; `Rear` là `a[-1]`. Không có "vòng" nào cả; ta chỉ **giới hạn độ dài** bằng `k`.

Cách này **đúng** về hành vi, nhưng không thật sự là "hàng đợi vòng" và `deQueue` tốn `O(n)`.

In [9]:
class NaiveQueue:
    def __init__(self, k):
        self.k = k
        self.a = []

    def enQueue(self, value):
        if len(self.a) == self.k:
            return False
        self.a.append(value)
        return True

    def deQueue(self):
        if not self.a:
            return False
        self.a.pop(0)               # O(n): dịch cả list
        return True

    def Front(self):
        return self.a[0] if self.a else -1

    def Rear(self):
        return self.a[-1] if self.a else -1

    def isEmpty(self):
        return len(self.a) == 0

    def isFull(self):
        return len(self.a) == self.k

got = run_ops(NaiveQueue, ex_ops, ex_args)
print(got)
assert got == ex_expected

[None, True, True, True, False, 3, True, True, True, 4]


**Độ phức tạp:** `enQueue`, `Front`, `Rear`, `isEmpty`, `isFull`: `O(1)`; `deQueue`: **`O(k)`** vì `pop(0)`. Space `O(k)`.

**Nút thắt:** với `k <= 1000` và 3000 lệnh thì vẫn qua được, nhưng đây là bài học về cấu trúc dữ liệu: ta muốn **mọi** thao tác `O(1)`. Cách khắc phục: không dịch phần tử mà **di chuyển con trỏ đầu hàng** trên một mảng cố định, dùng `% k` để quay vòng.

## 6. Ý tưởng thuật toán / bước đột phá

Dùng **mảng cố định `q` độ dài `k`** và nhớ hai con số:

- `front`: chỉ số của phần tử đầu hàng;
- `size`: số phần tử hiện có.

Khi đó (tất cả `O(1)`):

- ô thêm mới = `(front + size) % k`;
- ô cuối hàng (`Rear`) = `(front + size - 1) % k`;
- lấy ra: `front = (front + 1) % k`, `size -= 1`;
- rỗng: `size == 0`; đầy: `size == k`.

**Vì sao cần `size`?** Thử một thiết kế "ngây thơ" chỉ có hai con trỏ `head` (đầu) và `tail` (ô sẽ ghi tiếp theo). Khi hàng **rỗng**, `head == tail`. Nhưng khi hàng **đầy** (đã ghi `k` phần tử), `tail` cũng quay một vòng về đúng `head`, và lại `head == tail`! Hai trạng thái trái ngược cho cùng một dấu hiệu — không phân biệt được.

In [10]:
k = 3
head = tail = 0
print("rỗng :", "head =", head, "tail =", tail, "-> head == tail:", head == tail)
for v in (1, 2, 3):                 # thêm đủ k phần tử
    tail = (tail + 1) % k
print("đầy  :", "head =", head, "tail =", tail, "-> head == tail:", head == tail)
print("=> cùng điều kiện cho 'rỗng' và 'đầy': cần thêm thông tin (size) hoặc chừa 1 ô trống")

rỗng : head = 0 tail = 0 -> head == tail: True
đầy  : head = 0 tail = 0 -> head == tail: True
=> cùng điều kiện cho 'rỗng' và 'đầy': cần thêm thông tin (size) hoặc chừa 1 ô trống


Có hai cách giải quyết mơ hồ này:

1. **Thêm biến `size`** (Cách 2 ở mục 7): rỗng ⇔ `size == 0`, đầy ⇔ `size == k`. Đơn giản và dễ chứng minh.
2. **Chừa trống một ô** (Cách 3): mảng `k + 1` ô, quy ước "đầy" khi `(tail + 1) % (k + 1) == head`, như vậy `head == tail` chỉ còn nghĩa là rỗng.
3. (Biến thể) **Hai bộ đếm tăng mãi** (Cách 4): `size = tail - head` luôn rõ ràng, chỉ số mảng là `% k`.

## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 list + `pop(0)` (đã làm); Cách 2 mảng + `front` + `size` (cách chính); Cách 3 hai con trỏ với mảng `k + 1` ô; Cách 4 hai bộ đếm tăng mãi; Cách 5 danh sách liên kết với `head`, `tail`, `size`.

### 7.1. Cách 2: Mảng `k` ô + `front` + `size`

**Giải thích từng đoạn:**

1. `__init__`: `self.q = [0] * k`, `self.front = 0`, `self.size = 0`, `self.k = k`.
2. `enQueue`: nếu đầy trả `False`; ngược lại ghi vào ô `(front + size) % k`, tăng `size`, trả `True`.
3. `deQueue`: nếu rỗng trả `False`; ngược lại `front = (front + 1) % k`, giảm `size`, trả `True`. Ta **không cần** xóa giá trị cũ trong ô; nó sẽ bị ghi đè khi cần.
4. `Front`: rỗng trả `-1`, ngược lại `q[front]`.
5. `Rear`: rỗng trả `-1`, ngược lại `q[(front + size - 1) % k]`.
6. `isEmpty`: `size == 0`; `isFull`: `size == k`.

In [11]:
class SizeQueue:
    def __init__(self, k):
        self.q = [0] * k
        self.k = k
        self.front = 0
        self.size = 0

    def enQueue(self, value):
        if self.isFull():
            return False
        self.q[(self.front + self.size) % self.k] = value
        self.size += 1
        return True

    def deQueue(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True

    def Front(self):
        return -1 if self.isEmpty() else self.q[self.front]

    def Rear(self):
        if self.isEmpty():
            return -1
        return self.q[(self.front + self.size - 1) % self.k]

    def isEmpty(self):
        return self.size == 0

    def isFull(self):
        return self.size == self.k

got = run_ops(SizeQueue, ex_ops, ex_args)
print(got)
assert got == ex_expected

[None, True, True, True, False, 3, True, True, True, 4]


**Code nộp LeetCode (Cách 2):**

```python
class MyCircularQueue:

    def __init__(self, k: int):
        self.q = [0] * k
        self.k = k
        self.front = 0
        self.size = 0

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[(self.front + self.size) % self.k] = value
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.q[self.front]

    def Rear(self) -> int:
        if self.isEmpty():
            return -1
        return self.q[(self.front + self.size - 1) % self.k]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.k
```

**Độ phức tạp:** mọi thao tác `O(1)`; Space `O(k)`.

### 7.2. Cách 3: Hai con trỏ `head`/`tail`, mảng `k + 1` ô (chừa một ô trống)

**Giải thích từng đoạn:**

1. Mảng dài `n = k + 1`. `head` là ô đầu hàng, `tail` là **ô sẽ ghi tiếp theo** (ô ngay sau phần tử cuối).
2. **Rỗng:** `head == tail`.
3. **Đầy:** `(tail + 1) % n == head`: nếu ghi thêm nữa thì `tail` sẽ trùng `head`, nên ta để lại một ô không dùng, dùng nó làm "dấu" phân biệt đầy/rỗng. Sức chứa thực là `n - 1 = k`.
4. `enQueue`: ghi vào `q[tail]`, `tail = (tail + 1) % n`. `deQueue`: `head = (head + 1) % n`.
5. `Rear`: phần tử ngay **trước** `tail`: `q[(tail - 1) % n]`. Trong Python `%` luôn không âm nên được; trong C++/Java phải viết `(tail - 1 + n) % n`.

In [12]:
class TwoPointerQueue:
    def __init__(self, k):
        self.n = k + 1                  # chừa 1 ô trống
        self.q = [0] * self.n
        self.head = 0
        self.tail = 0

    def enQueue(self, value):
        if self.isFull():
            return False
        self.q[self.tail] = value
        self.tail = (self.tail + 1) % self.n
        return True

    def deQueue(self):
        if self.isEmpty():
            return False
        self.head = (self.head + 1) % self.n
        return True

    def Front(self):
        return -1 if self.isEmpty() else self.q[self.head]

    def Rear(self):
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.n]

    def isEmpty(self):
        return self.head == self.tail

    def isFull(self):
        return (self.tail + 1) % self.n == self.head

got = run_ops(TwoPointerQueue, ex_ops, ex_args)
print(got)
assert got == ex_expected

[None, True, True, True, False, 3, True, True, True, 4]


**Code nộp LeetCode (Cách 3):**

```python
class MyCircularQueue:

    def __init__(self, k: int):
        self.n = k + 1
        self.q = [0] * self.n
        self.head = 0
        self.tail = 0

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[self.tail] = value
        self.tail = (self.tail + 1) % self.n
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.head = (self.head + 1) % self.n
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.q[self.head]

    def Rear(self) -> int:
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.n]

    def isEmpty(self) -> bool:
        return self.head == self.tail

    def isFull(self) -> bool:
        return (self.tail + 1) % self.n == self.head
```

**Độ phức tạp:** mọi thao tác `O(1)`; Space `O(k + 1) = O(k)`. Ưu điểm: không cần `size`; nhược: tốn thêm một ô và dễ nhầm điều kiện đầy.

### 7.3. Cách 4: Hai bộ đếm tăng mãi

**Ý tưởng:** thay vì giữ chỉ số đã `% k`, ta giữ hai **bộ đếm không bao giờ quay vòng**: `head` = tổng số lần đã `deQueue`, `tail` = tổng số lần đã `enQueue`. Khi đó **số phần tử = `tail - head`** (rõ ràng, không mơ hồ), và chỉ khi truy cập mảng mới lấy `% k`.

**Giải thích từng đoạn:**

1. Rỗng: `tail == head`; đầy: `tail - head == k`.
2. `enQueue`: ghi `q[tail % k]`, `tail += 1`. `deQueue`: `head += 1`.
3. `Front = q[head % k]`, `Rear = q[(tail - 1) % k]`.
4. Python có số nguyên lớn tùy ý nên không tràn; ngôn ngữ khác phải cẩn thận kiểu số (tràn `int` sau hơn 2 tỉ lệnh; ở bài này chỉ có 3000 lệnh).

In [13]:
class CounterQueue:
    def __init__(self, k):
        self.k = k
        self.q = [0] * k
        self.head = 0       # tổng số lần deQueue thành công
        self.tail = 0       # tổng số lần enQueue thành công

    def enQueue(self, value):
        if self.isFull():
            return False
        self.q[self.tail % self.k] = value
        self.tail += 1
        return True

    def deQueue(self):
        if self.isEmpty():
            return False
        self.head += 1
        return True

    def Front(self):
        return -1 if self.isEmpty() else self.q[self.head % self.k]

    def Rear(self):
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.k]

    def isEmpty(self):
        return self.tail == self.head

    def isFull(self):
        return self.tail - self.head == self.k

got = run_ops(CounterQueue, ex_ops, ex_args)
print(got)
assert got == ex_expected

[None, True, True, True, False, 3, True, True, True, 4]


**Code nộp LeetCode (Cách 4):**

```python
class MyCircularQueue:

    def __init__(self, k: int):
        self.k = k
        self.q = [0] * k
        self.head = 0
        self.tail = 0

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[self.tail % self.k] = value
        self.tail += 1
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.head += 1
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.q[self.head % self.k]

    def Rear(self) -> int:
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.k]

    def isEmpty(self) -> bool:
        return self.tail == self.head

    def isFull(self) -> bool:
        return self.tail - self.head == self.k
```

**Độ phức tạp:** mọi thao tác `O(1)`; Space `O(k)`.

### 7.4. Cách 5: Danh sách liên kết đơn với `head`, `tail`, `size`

**Ý tưởng khác hẳn:** không dùng mảng cố định. Lưu các phần tử trong **danh sách liên kết**, giữ `head`, `tail` và `size`; "vòng tròn" ở đây chỉ còn là việc giới hạn sức chứa `k`.

**Giải thích từng đoạn:**

1. `enQueue`: nếu `size == k` trả `False`. Tạo nút mới; nếu rỗng thì `head = tail = nút`, ngược lại `tail.next = nút; tail = nút`. Tăng `size`.
2. `deQueue`: nếu rỗng trả `False`; `head = head.next`, giảm `size`; nếu thành rỗng thì **đặt `tail = None`** để không giữ nút cũ.
3. `Front = head.val`, `Rear = tail.val`.

Mỗi thao tác `O(1)`; Space `O(k)` nhưng mỗi nút tốn thêm một tham chiếu.

In [14]:
class LinkedQueue:
    class _Node:
        def __init__(self, val):
            self.val = val
            self.next = None

    def __init__(self, k):
        self.k = k
        self.head = None
        self.tail = None
        self.size = 0

    def enQueue(self, value):
        if self.isFull():
            return False
        node = self._Node(value)
        if self.size == 0:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self.size += 1
        return True

    def deQueue(self):
        if self.isEmpty():
            return False
        self.head = self.head.next
        self.size -= 1
        if self.size == 0:
            self.tail = None
        return True

    def Front(self):
        return -1 if self.isEmpty() else self.head.val

    def Rear(self):
        return -1 if self.isEmpty() else self.tail.val

    def isEmpty(self):
        return self.size == 0

    def isFull(self):
        return self.size == self.k

got = run_ops(LinkedQueue, ex_ops, ex_args)
print(got)
assert got == ex_expected

[None, True, True, True, False, 3, True, True, True, 4]


**Code nộp LeetCode (Cách 5):**

```python
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None


class MyCircularQueue:

    def __init__(self, k: int):
        self.k = k
        self.head = None
        self.tail = None
        self.size = 0

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        node = Node(value)
        if self.size == 0:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.head = self.head.next
        self.size -= 1
        if self.size == 0:
            self.tail = None
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.head.val

    def Rear(self) -> int:
        return -1 if self.isEmpty() else self.tail.val

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.k
```

**Độ phức tạp:** mọi thao tác `O(1)`; Space `O(k)`.

## 8. TRACE từng bước

Ta mô phỏng Cách 2 (mảng + `front` + `size`) với ví dụ 1 (`k = 3`). Trạng thái ban đầu: `q = [0, 0, 0]`, `front = 0`, `size = 0`.

| # | Lệnh | Tính toán | `q` | `front` | `size` | Kết quả |
|---|---|---|---|---|---|---|
| 1 | `enQueue(1)` | ô `(0+0)%3 = 0` | `[1,0,0]` | 0 | 1 | `True` |
| 2 | `enQueue(2)` | ô `(0+1)%3 = 1` | `[1,2,0]` | 0 | 2 | `True` |
| 3 | `enQueue(3)` | ô `(0+2)%3 = 2` | `[1,2,3]` | 0 | 3 | `True` |
| 4 | `enQueue(4)` | `size == 3` đầy | `[1,2,3]` | 0 | 3 | `False` |
| 5 | `Rear()` | ô `(0+3-1)%3 = 2` | | 0 | 3 | `3` |
| 6 | `isFull()` | `3 == 3` | | 0 | 3 | `True` |
| 7 | `deQueue()` | `front = (0+1)%3 = 1` | `[1,2,3]` | 1 | 2 | `True` |
| 8 | `enQueue(4)` | ô `(1+2)%3 = 0` | `[4,2,3]` | 1 | 3 | `True` |
| 9 | `Rear()` | ô `(1+3-1)%3 = 0` | | 1 | 3 | `4` |

Ở bước 7, giá trị `1` vẫn còn trong ô 0 (không bị xóa), nhưng nó **nằm ngoài** vùng logic nên bị bỏ qua; bước 8 ghi đè lên đúng ô đó.

In [15]:
def trace_size_queue(k, ops, args):
    q = [0] * k
    front = size = 0
    print(f"khởi tạo: q={q}, front={front}, size={size}")
    for step, (name, a) in enumerate(zip(ops[1:], args[1:]), 1):
        if name == "enQueue":
            if size == k:
                res = False
                note = "đầy (size == k)"
            else:
                pos = (front + size) % k
                q[pos] = a[0]
                size += 1
                res = True
                note = f"ghi vào ô (front+size)%k = {pos}"
        elif name == "deQueue":
            if size == 0:
                res = False
                note = "rỗng"
            else:
                front = (front + 1) % k
                size -= 1
                res = True
                note = f"front -> {front}"
        elif name == "Front":
            res = -1 if size == 0 else q[front]
            note = f"q[front] = q[{front}]"
        elif name == "Rear":
            if size == 0:
                res, note = -1, "rỗng"
            else:
                pos = (front + size - 1) % k
                res, note = q[pos], f"q[(front+size-1)%k] = q[{pos}]"
        elif name == "isEmpty":
            res, note = size == 0, f"size == 0"
        else:
            res, note = size == k, f"size == k"
        logical = [q[(front + i) % k] for i in range(size)]
        print(f"#{step} {name}{tuple(a) if a else '()'} -> {res!s:5} | {note:34} | q={q} front={front} size={size} | hàng đợi={logical}")

trace_size_queue(3, ex_ops, ex_args)

khởi tạo: q=[0, 0, 0], front=0, size=0
#1 enQueue(1,) -> True  | ghi vào ô (front+size)%k = 0       | q=[1, 0, 0] front=0 size=1 | hàng đợi=[1]
#2 enQueue(2,) -> True  | ghi vào ô (front+size)%k = 1       | q=[1, 2, 0] front=0 size=2 | hàng đợi=[1, 2]
#3 enQueue(3,) -> True  | ghi vào ô (front+size)%k = 2       | q=[1, 2, 3] front=0 size=3 | hàng đợi=[1, 2, 3]
#4 enQueue(4,) -> False | đầy (size == k)                    | q=[1, 2, 3] front=0 size=3 | hàng đợi=[1, 2, 3]
#5 Rear() -> 3     | q[(front+size-1)%k] = q[2]         | q=[1, 2, 3] front=0 size=3 | hàng đợi=[1, 2, 3]
#6 isFull() -> True  | size == k                          | q=[1, 2, 3] front=0 size=3 | hàng đợi=[1, 2, 3]
#7 deQueue() -> True  | front -> 1                         | q=[1, 2, 3] front=1 size=2 | hàng đợi=[2, 3]
#8 enQueue(4,) -> True  | ghi vào ô (front+size)%k = 0       | q=[4, 2, 3] front=1 size=3 | hàng đợi=[2, 3, 4]
#9 Rear() -> 4     | q[(front+size-1)%k] = q[0]         | q=[4, 2, 3] front=1 size=3 | hàng đợi

Thử thêm một chuỗi lệnh **quay vòng nhiều lần** với `k = 2` để thấy `front` đi `0 → 1 → 0 → 1`:

In [16]:
ops2 = ["MyCircularQueue"] + ["enQueue", "enQueue", "deQueue", "enQueue", "deQueue", "enQueue", "Front", "Rear", "deQueue", "deQueue", "deQueue", "Front"]
args2 = [[2], [10], [20], [], [30], [], [40], [], [], [], [], [], []]
trace_size_queue(2, ops2, args2)

khởi tạo: q=[0, 0], front=0, size=0
#1 enQueue(10,) -> True  | ghi vào ô (front+size)%k = 0       | q=[10, 0] front=0 size=1 | hàng đợi=[10]
#2 enQueue(20,) -> True  | ghi vào ô (front+size)%k = 1       | q=[10, 20] front=0 size=2 | hàng đợi=[10, 20]
#3 deQueue() -> True  | front -> 1                         | q=[10, 20] front=1 size=1 | hàng đợi=[20]
#4 enQueue(30,) -> True  | ghi vào ô (front+size)%k = 0       | q=[30, 20] front=1 size=2 | hàng đợi=[20, 30]
#5 deQueue() -> True  | front -> 0                         | q=[30, 20] front=0 size=1 | hàng đợi=[30]
#6 enQueue(40,) -> True  | ghi vào ô (front+size)%k = 1       | q=[30, 40] front=0 size=2 | hàng đợi=[30, 40]
#7 Front() -> 30    | q[front] = q[0]                    | q=[30, 40] front=0 size=2 | hàng đợi=[30, 40]
#8 Rear() -> 40    | q[(front+size-1)%k] = q[1]         | q=[30, 40] front=0 size=2 | hàng đợi=[30, 40]
#9 deQueue() -> True  | front -> 1                         | q=[30, 40] front=1 size=1 | hàng đợi=[40]
#10 deQueue

Quan sát: `front` quay lại `0` sau hai lần `deQueue`, và các giá trị mới lần lượt ghi đè các ô đã được trả ra. Khi rỗng, `Front()` trả `-1` dù ô nhớ vẫn còn dữ liệu cũ; chính `size == 0` mới quyết định.

## 9. Code nộp LeetCode

Ta nộp **Cách 2** (mảng + `front` + `size`) vì ngắn, rõ, ít chỗ sai nhất.

In [17]:
class MyCircularQueue:

    def __init__(self, k: int):
        self.q = [0] * k          # mảng cố định k ô
        self.k = k
        self.front = 0            # chỉ số phần tử đầu hàng
        self.size = 0             # số phần tử hiện có

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[(self.front + self.size) % self.k] = value
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.q[self.front]

    def Rear(self) -> int:
        if self.isEmpty():
            return -1
        return self.q[(self.front + self.size - 1) % self.k]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.k


got = run_ops(MyCircularQueue, ex_ops, ex_args)
print("kết quả :", got)
print("kỳ vọng :", ex_expected)
assert got == ex_expected

kết quả : [None, True, True, True, False, 3, True, True, True, 4]
kỳ vọng : [None, True, True, True, False, 3, True, True, True, 4]


## 10. Vì sao đúng

Ta chứng minh bằng một **bất biến (invariant)**: sau mỗi thao tác, hàng đợi logic là dãy `q[front], q[(front+1)%k], ..., q[(front+size-1)%k]` (đúng `size` phần tử, theo thứ tự vào), và `0 <= size <= k`, `0 <= front < k`.

- **Ban đầu:** `size = 0`, hàng rỗng, bất biến đúng.
- **`enQueue`** (khi `size < k`): ô `(front+size)%k` không nằm trong `size` ô logic hiện tại. Vì các ô logic chiếm `size` vị trí liên tiếp (theo vòng) bắt đầu từ `front`, và `size < k` nên còn ít nhất một ô; ô ngay sau ô cuối chính là `(front+size)%k`. Ghi vào đó rồi `size += 1` làm hàng logic dài thêm đúng một phần tử ở **cuối**. Đúng FIFO.
- **`deQueue`** (khi `size > 0`): `front` tiến một ô, `size` giảm 1, vậy phần tử đầu bị loại, các phần tử còn lại giữ nguyên thứ tự.
- **`Front`, `Rear`:** đọc ô đầu hoặc ô cuối của hàng logic.
- **`isEmpty`, `isFull`:** vì `size` chính xác bằng số phần tử, đây là câu trả lời đúng; không có mơ hồ như hai con trỏ.
- **Độ lớn:** `enQueue` chỉ chạy khi `size < k` nên `size <= k` mãi; `deQueue` chỉ khi `size > 0` nên `size >= 0`.

Ta kiểm tra bất biến bằng code: sau **mỗi** lệnh trong 3000 lệnh ngẫu nhiên, hàng logic của `MyCircularQueue` phải bằng hàng đợi tham chiếu (`collections.deque` chỉ dùng trong **kiểm tra**, không dùng trong lời giải).

In [18]:
def logical_content(obj):
    return [obj.q[(obj.front + i) % obj.k] for i in range(obj.size)]

rng = random.Random(622)
for k in (1, 2, 3, 7):
    obj = MyCircularQueue(k)
    ref = deque()
    for _ in range(3000):
        action = rng.choice(["enQueue", "enQueue", "deQueue", "Front", "Rear"])
        if action == "enQueue":
            v = rng.randint(0, 1000)
            ok = obj.enQueue(v)
            if len(ref) < k:
                ref.append(v)
                assert ok
            else:
                assert not ok
        elif action == "deQueue":
            ok = obj.deQueue()
            assert ok == bool(ref)
            if ref:
                ref.popleft()
        elif action == "Front":
            assert obj.Front() == (ref[0] if ref else -1)
        else:
            assert obj.Rear() == (ref[-1] if ref else -1)
        # bất biến
        assert 0 <= obj.size <= k and 0 <= obj.front < k
        assert logical_content(obj) == list(ref)
print("bất biến giữ đúng sau mọi lệnh với k = 1, 2, 3, 7")

bất biến giữ đúng sau mọi lệnh với k = 1, 2, 3, 7


## 11. Time/Space Complexity

| Cách | `enQueue` | `deQueue` | `Front`/`Rear` | `isEmpty`/`isFull` | Space |
|---|---|---|---|---|---|
| 1. List + `pop(0)` | `O(1)` | **`O(k)`** | `O(1)` | `O(1)` | `O(k)` |
| 2. Mảng + `front` + `size` | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(k)` |
| 3. Hai con trỏ, mảng `k+1` | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(k)` |
| 4. Hai bộ đếm tăng mãi | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(k)` |
| 5. Danh sách liên kết | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(k)` (thêm con trỏ) |

Với `Q` lệnh, tổng thời gian là `O(Q)` cho Cách 2 đến 5 (và `O(Q * k)` tệ nhất cho Cách 1). Bộ nhớ không phụ thuộc số lệnh, chỉ phụ thuộc `k`.

## 12. Bộ test

Các nhóm test: ví dụ đề, edge case (`k = 1`, dequeue/Front/Rear khi rỗng, đầy rồi đầy lại), phản ví dụ cho các cách làm sai, và quan trọng nhất là **đối chiếu 5 cách + lớp nộp với oracle hàng đợi tham chiếu** trên chuỗi lệnh ngẫu nhiên có seed, đúng ràng buộc (tối đa 3000 lệnh, `k` tới 1000, `value` trong `[0, 1000]`).

In [19]:
# 12.1 ví dụ của đề và edge case
for cls in (NaiveQueue, SizeQueue, TwoPointerQueue, CounterQueue, LinkedQueue, MyCircularQueue):
    assert run_ops(cls, ex_ops, ex_args) == ex_expected, cls.__name__

# k = 1: sức chứa nhỏ nhất
ops = ["MyCircularQueue", "isEmpty", "isFull", "Front", "Rear", "deQueue", "enQueue", "enQueue", "Front", "Rear", "isFull", "deQueue", "isEmpty", "enQueue", "Rear"]
args = [[1], [], [], [], [], [], [5], [6], [], [], [], [], [], [0], []]
expected = [None, True, False, -1, -1, False, True, False, 5, 5, True, True, True, True, 0]
for cls in (NaiveQueue, SizeQueue, TwoPointerQueue, CounterQueue, LinkedQueue, MyCircularQueue):
    assert run_ops(cls, ops, args) == expected, cls.__name__
print("ví dụ đề + k = 1: OK")

# thao tác trên hàng đợi rỗng
ops = ["MyCircularQueue", "deQueue", "Front", "Rear", "isEmpty", "isFull"]
args = [[3], [], [], [], [], []]
expected = [None, False, -1, -1, True, False]
for cls in (NaiveQueue, SizeQueue, TwoPointerQueue, CounterQueue, LinkedQueue, MyCircularQueue):
    assert run_ops(cls, ops, args) == expected, cls.__name__

# đầy rồi cạn rồi đầy lại, giá trị 0 hợp lệ (khác với -1)
ops = ["MyCircularQueue"] + ["enQueue"] * 4 + ["isFull"] + ["deQueue"] * 4 + ["isEmpty", "Front"] + ["enQueue"] * 2 + ["Front", "Rear"]
args = [[3], [0], [0], [0], [9], [], [], [], [], [], [], [], [7], [8], [], []]
expected = [None, True, True, True, False, True, True, True, True, False, True, -1, True, True, 7, 8]
for cls in (NaiveQueue, SizeQueue, TwoPointerQueue, CounterQueue, LinkedQueue, MyCircularQueue):
    assert run_ops(cls, ops, args) == expected, (cls.__name__, run_ops(cls, ops, args))
print("rỗng / đầy / giá trị 0: OK")

ví dụ đề + k = 1: OK
rỗng / đầy / giá trị 0: OK


In [20]:
# 12.2 phản ví dụ: các cài đặt SAI phổ biến
class WrongNoMod(SizeQueue):             # quên % k khi ghi và đọc Rear
    def enQueue(self, value):
        if self.isFull():
            return False
        self.q[self.front + self.size] = value        # SAI: có thể vượt chỉ số
        self.size += 1
        return True

class WrongHeadTail:                      # hai con trỏ, mảng k ô: nhầm rỗng/đầy
    def __init__(self, k):
        self.k, self.q, self.head, self.tail = k, [0] * k, 0, 0
    def enQueue(self, v):
        if self.isFull():
            return False
        self.q[self.tail] = v
        self.tail = (self.tail + 1) % self.k
        return True
    def deQueue(self):
        if self.isEmpty():
            return False
        self.head = (self.head + 1) % self.k
        return True
    def Front(self):
        return -1 if self.isEmpty() else self.q[self.head]
    def Rear(self):
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.k]
    def isEmpty(self):
        return self.head == self.tail
    def isFull(self):
        return self.head == self.tail                 # SAI: trùng điều kiện với isEmpty

class WrongForgetSize(SizeQueue):         # deQueue quên giảm size
    def deQueue(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.k
        return True

res_nomod = None
try:
    res_nomod = run_ops(WrongNoMod, ex_ops, ex_args)
except IndexError as e:
    res_nomod = "IndexError: " + str(e)
print("quên % k       :", res_nomod)
print("head==tail cho cả rỗng và đầy:", run_ops(WrongHeadTail, ex_ops, ex_args))
print("quên giảm size :", run_ops(WrongForgetSize, ex_ops, ex_args))
print("đúng           :", ex_expected)
assert res_nomod != ex_expected
assert run_ops(WrongHeadTail, ex_ops, ex_args) != ex_expected
assert run_ops(WrongForgetSize, ex_ops, ex_args) != ex_expected

quên % k       : IndexError: list assignment index out of range
head==tail cho cả rỗng và đầy: [None, False, False, False, False, -1, True, False, False, -1]
quên giảm size : [None, True, True, True, False, 3, True, True, False, 1]
đúng           : [None, True, True, True, False, 3, True, True, True, 4]


In [21]:
# 12.3 đối chiếu mọi cài đặt với oracle (deque tham chiếu) trên chuỗi lệnh ngẫu nhiên có seed
def oracle_run(ops, args):
    out, ref, k = [], deque(), None
    for i, (name, a) in enumerate(zip(ops, args)):
        if i == 0:
            k = a[0]
            out.append(None)
        elif name == "enQueue":
            if len(ref) < k:
                ref.append(a[0]); out.append(True)
            else:
                out.append(False)
        elif name == "deQueue":
            if ref:
                ref.popleft(); out.append(True)
            else:
                out.append(False)
        elif name == "Front":
            out.append(ref[0] if ref else -1)
        elif name == "Rear":
            out.append(ref[-1] if ref else -1)
        elif name == "isEmpty":
            out.append(len(ref) == 0)
        else:
            out.append(len(ref) == k)
    return out

NAMES = ["enQueue", "deQueue", "Front", "Rear", "isEmpty", "isFull"]

def random_ops(rng, k, calls, p_enq):
    ops, args = ["MyCircularQueue"], [[k]]
    for _ in range(calls):
        r = rng.random()
        if r < p_enq:
            ops.append("enQueue"); args.append([rng.randint(0, 1000)])
        else:
            name = rng.choice(["deQueue", "deQueue", "Front", "Rear", "isEmpty", "isFull"])
            ops.append(name); args.append([])
    return ops, args

impls = [NaiveQueue, SizeQueue, TwoPointerQueue, CounterQueue, LinkedQueue, MyCircularQueue]
rng = random.Random(2022)
total_calls = 0
for trial in range(300):
    k = rng.choice([1, 2, 3, 4, 5, 8, 13, 50])
    calls = rng.randint(1, 200)
    ops, args = random_ops(rng, k, calls, p_enq=rng.choice([0.3, 0.5, 0.7]))
    expected = oracle_run(ops, args)
    for cls in impls:
        assert run_ops(cls, ops, args) == expected, (cls.__name__, k, trial)
    total_calls += len(ops)
print(f"300 chuỗi lệnh ngẫu nhiên ({total_calls} lệnh): cả {len(impls)} cài đặt khớp oracle")

300 chuỗi lệnh ngẫu nhiên (30953 lệnh): cả 6 cài đặt khớp oracle


In [22]:
# 12.4 test lớn đúng ràng buộc: k = 1000, 3000 lệnh
rng = random.Random(99)
for p_enq in (0.9, 0.5, 0.1):
    ops, args = random_ops(rng, 1000, 3000 - 1, p_enq)
    assert len(ops) == 3000
    expected = oracle_run(ops, args)
    for cls in impls:
        assert run_ops(cls, ops, args) == expected, (cls.__name__, p_enq)

# nhồi đầy đến k = 1000 rồi cạn: kiểm tra biên đầy tuyệt đối
ops = ["MyCircularQueue"] + ["enQueue"] * 1001 + ["isFull", "Front", "Rear"] + ["deQueue"] * 1001 + ["isEmpty"]
args = [[1000]] + [[i % 1001] for i in range(1001)] + [[], [], []] + [[]] * 1001 + [[]]
res = run_ops(MyCircularQueue, ops, args)
assert res[1:1001] == [True] * 1000 and res[1001] is False
assert res[1002:1005] == [True, 0, 999] and res[-2] is False and res[-1] is True
assert res == oracle_run(ops, args)
print("test lớn (k = 1000, 3000 lệnh) và biên đầy tuyệt đối: OK")

# thời gian: 3000 lệnh nhanh với mọi cách O(1); Cách 1 vẫn đủ nhanh vì k nhỏ
ops, args = random_ops(rng, 1000, 2999, 0.5)
for cls in impls:
    t0 = time.perf_counter()
    for _ in range(20):
        run_ops(cls, ops, args)
    print(f"{cls.__name__:16s} {(time.perf_counter() - t0) / 20 * 1000:6.2f} ms / 3000 lệnh")

test lớn (k = 1000, 3000 lệnh) và biên đầy tuyệt đối: OK
NaiveQueue         0.69 ms / 3000 lệnh


SizeQueue          1.03 ms / 3000 lệnh


TwoPointerQueue    1.70 ms / 3000 lệnh
CounterQueue       1.50 ms / 3000 lệnh
LinkedQueue        1.69 ms / 3000 lệnh
MyCircularQueue    1.52 ms / 3000 lệnh


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `% k` khi tính vị trí ghi hoặc `Rear`

`front + size` có thể bằng `k` hoặc lớn hơn. Không quay vòng sẽ gây `IndexError` hoặc ghi sai ô.

In [23]:
class Bug1:
    def __init__(self, k):
        self.q, self.k, self.front, self.size = [0] * k, k, 0, 0
    def enQueue(self, value):
        if self.size == self.k:
            return False
        self.q[self.front + self.size] = value       # SAI: thiếu % k
        self.size += 1
        return True
    def deQueue(self):
        if self.size == 0:
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True

b = Bug1(3)
print([b.enQueue(1), b.enQueue(2), b.enQueue(3), b.deQueue()])
try:
    b.enQueue(4)
except IndexError as e:
    print("IndexError:", e, "-> ô (front+size) = 1+2 = 3 vượt k = 3")

[True, True, True, True]
IndexError: list assignment index out of range -> ô (front+size) = 1+2 = 3 vượt k = 3


**Vì sao sai:** sau `deQueue`, `front = 1`, `size = 2` nên ô ghi là `1 + 2 = 3`, mà mảng chỉ có ô `0..2`. **Sửa:** `(self.front + self.size) % self.k`.

### Lỗi 2: Hai con trỏ, mảng `k` ô, nhầm rỗng với đầy

Khi `head == tail` ta không biết hàng đợi rỗng hay đầy. Nếu dùng `head == tail` làm điều kiện **cả hai**, thì `isFull()` luôn bằng `isEmpty()`.

In [24]:
wrong = run_ops(WrongHeadTail, ex_ops, ex_args)
print("SAI  :", wrong)
print("ĐÚNG :", ex_expected)

t = WrongHeadTail(2)
print("hàng rỗng: isEmpty =", t.isEmpty(), "| isFull =", t.isFull(), "(cả hai True!)")

SAI  : [None, False, False, False, False, -1, True, False, False, -1]
ĐÚNG : [None, True, True, True, False, 3, True, True, True, 4]
hàng rỗng: isEmpty = True | isFull = True (cả hai True!)


**Vì sao sai:** hàng đợi mới tạo có `head == tail == 0`, nên bị coi là đầy luôn. **Sửa:** thêm `size`, hoặc dùng mảng `k + 1` ô và điều kiện đầy `(tail + 1) % (k + 1) == head`, hoặc dùng hai bộ đếm tăng mãi.

### Lỗi 3: Quên cập nhật `size` (hoặc cập nhật sai chiều)

In [25]:
print("SAI  :", run_ops(WrongForgetSize, ex_ops, ex_args))
print("ĐÚNG :", ex_expected)

SAI  : [None, True, True, True, False, 3, True, True, False, 1]
ĐÚNG : [None, True, True, True, False, 3, True, True, True, 4]


**Vì sao sai:** nếu `deQueue` không giảm `size` thì hàng đợi không bao giờ "bớt" phần tử; `enQueue(4)` ở bước 8 bị từ chối (`False`) và `Rear` lệch. **Sửa:** mỗi thao tác phải cập nhật **đủ** các biến trạng thái (`front` và `size`).

### Lỗi 4: Tính `Rear` bằng `q[front + size]` thay vì `q[front + size - 1]`

`(front + size) % k` là ô **sẽ ghi tiếp**, không phải ô cuối hàng. Ô cuối hàng lùi một ô: `(front + size - 1) % k`.

In [26]:
class Bug4(SizeQueue):
    def Rear(self):
        if self.isEmpty():
            return -1
        return self.q[(self.front + self.size) % self.k]   # SAI: ô kế tiếp, không phải ô cuối

print("SAI  :", run_ops(Bug4, ex_ops, ex_args))
print("ĐÚNG :", ex_expected)
assert run_ops(Bug4, ex_ops, ex_args) != ex_expected

SAI  : [None, True, True, True, False, 1, True, True, True, 2]
ĐÚNG : [None, True, True, True, False, 3, True, True, True, 4]


**Vì sao sai:** ở bước 5 (`Rear` khi đã đầy), `(0 + 3) % 3 = 0` trỏ vào phần tử **đầu** (`1`), không phải `3`. **Sửa:** `- 1`.

### Lỗi 5: Dùng `-1` làm "ô trống" trong mảng rồi nhầm với dữ liệu

Ở bài này `value >= 0` nên `-1` an toàn làm giá trị báo rỗng của `Front`/`Rear`. Nhưng đừng dựa vào nội dung ô để suy ra rỗng/đầy; hãy dựa vào `size` (hoặc con trỏ). Giá trị `0` là dữ liệu hợp lệ (đã test ở mục 12).

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- Đề nói "**ring buffer**", "**circular**", "**sliding window cố định kích thước**", "**bộ đệm có giới hạn**", hoặc "ghi đè phần tử cũ nhất".
- Cần thao tác đầu/cuối `O(1)` trên cấu trúc có kích thước **cố định**.

**Mẹo:**

1. **Chọn trạng thái tối thiểu mà vẫn không mơ hồ**: `front` + `size` là lựa chọn "an toàn" nhất.
2. Ba công thức luôn nhớ: ghi `(front + size) % k`, cuối `(front + size - 1) % k`, lấy ra `front = (front + 1) % k`.
3. Khi lùi một ô trong ngôn ngữ khác Python, viết `(i - 1 + k) % k` để tránh số âm.
4. Viết **bộ chạy chuỗi lệnh** và **oracle** bằng `deque` để so sánh ngẫu nhiên; bài thiết kế rất hợp với kiểu test này.
5. Không cần xóa dữ liệu cũ khi dequeue; ô đó sẽ bị ghi đè khi cần (lazy deletion).
6. Với danh sách liên kết, nhớ **đặt `tail = None`** khi hàng đợi về rỗng.

## 15. Biến thể

### 15.1. Hàng đợi hai đầu vòng (LeetCode 641 — Design Circular Deque)

Thêm/xóa được ở **cả hai đầu**. Dùng cùng khung `front` + `size`: `insertLast` ghi tại `(front + size) % k`; `insertFront` **lùi `front`** một ô `(front - 1) % k` rồi ghi; `deleteFront` tiến `front`; `deleteLast` chỉ giảm `size`.

In [27]:
class CircularDeque:
    def __init__(self, k):
        self.q, self.k, self.front, self.size = [0] * k, k, 0, 0
    def insertFront(self, v):
        if self.size == self.k:
            return False
        self.front = (self.front - 1) % self.k
        self.q[self.front] = v
        self.size += 1
        return True
    def insertLast(self, v):
        if self.size == self.k:
            return False
        self.q[(self.front + self.size) % self.k] = v
        self.size += 1
        return True
    def deleteFront(self):
        if self.size == 0:
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True
    def deleteLast(self):
        if self.size == 0:
            return False
        self.size -= 1
        return True
    def getFront(self):
        return -1 if self.size == 0 else self.q[self.front]
    def getRear(self):
        return -1 if self.size == 0 else self.q[(self.front + self.size - 1) % self.k]

d = CircularDeque(3)
print(d.insertLast(1), d.insertLast(2), d.insertFront(3), d.insertFront(4))   # True True True False
print(d.getRear(), d.getFront())                                              # 2 3
print(d.deleteLast(), d.getRear(), d.insertFront(4), d.getFront())            # True 1 True 4

True True True False
2 3
True 1 True 4


### 15.2. Bộ đệm vòng **ghi đè** khi đầy

Trong thực tế (âm thanh, log, "N giá trị gần nhất"), khi đầy ta **ghi đè phần tử cũ nhất** thay vì từ chối. Chỉ cần đổi `enQueue`: nếu đầy thì ghi vào `front`, rồi `front` tiến một ô (`size` giữ nguyên `k`).

In [28]:
class OverwriteRing:
    def __init__(self, k):
        self.q, self.k, self.front, self.size = [0] * k, k, 0, 0
    def push(self, v):
        if self.size < self.k:
            self.q[(self.front + self.size) % self.k] = v
            self.size += 1
        else:
            self.q[self.front] = v                   # ghi đè phần tử cũ nhất
            self.front = (self.front + 1) % self.k
    def items(self):
        return [self.q[(self.front + i) % self.k] for i in range(self.size)]

r = OverwriteRing(3)
for v in range(1, 8):
    r.push(v)
    print(f"push({v}) -> {r.items()}")
assert r.items() == [5, 6, 7]

push(1) -> [1]
push(2) -> [1, 2]
push(3) -> [1, 2, 3]
push(4) -> [2, 3, 4]
push(5) -> [3, 4, 5]
push(6) -> [4, 5, 6]
push(7) -> [5, 6, 7]


### 15.3. Các bài cùng họ

- **LeetCode 641 — Design Circular Deque:** hàng đợi vòng hai đầu (15.1).
- **LeetCode 346 — Moving Average from Data Stream:** trung bình `n` giá trị gần nhất, dùng ring buffer + tổng chạy.
- **LeetCode 933 — Number of Recent Calls:** hàng đợi, bỏ phần tử quá hạn từ đầu.
- **LeetCode 232 / 225 — Queue using Stacks / Stack using Queues:** đổi qua lại hai cấu trúc.
- **LeetCode 1670 — Design Front Middle Back Queue:** hai đầu + phần giữa.

## 16. So sánh các cách

| Cách | Ý tưởng | Time mỗi thao tác | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List + `pop(0)` | list thường, giới hạn độ dài | `O(k)` cho dequeue, còn lại `O(1)` | `O(k)` | chỉ để làm baseline / hiểu vì sao cần vòng |
| 2. Mảng + `front` + `size` | `% k`, `size` phân biệt rỗng/đầy | `O(1)` | `O(k)` | **cách nộp chuẩn**, dễ chứng minh |
| 3. Hai con trỏ, mảng `k+1` | chừa một ô để phân biệt đầy | `O(1)` | `O(k + 1)` | thiết kế ring buffer kinh điển (phần cứng, kernel) |
| 4. Hai bộ đếm tăng mãi | `size = tail - head`, `% k` khi truy cập | `O(1)` | `O(k)` | khi muốn bất biến đơn giản; coi chừng tràn số ở ngôn ngữ khác |
| 5. Danh sách liên kết | nút `next`, giữ `head`/`tail`/`size` | `O(1)` | `O(k)` + con trỏ | khi cấp phát linh hoạt, không cần mảng lớn trước |

Kết luận: để thi và phỏng vấn, chọn Cách 2; biết Cách 3 vì nó hay được hỏi (vì sao "chừa một ô").

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **không được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, đoán 8 giá trị được in ra:

```python
q = MyCircularQueue(2)
print(q.enQueue(5), q.enQueue(6), q.enQueue(7), q.Front())
print(q.deQueue(), q.enQueue(7), q.Rear(), q.Front())
```

### Bài 2 (tính tay)

Với `k = 4`, thực hiện lần lượt: `enQueue(1)`, `enQueue(2)`, `enQueue(3)`, `enQueue(4)`, `deQueue()`, `deQueue()`, `enQueue(5)`, `enQueue(6)`. Hãy viết ra nội dung 4 ô `q`, giá trị `front`, `size`, và `Front()`, `Rear()` sau chuỗi lệnh.

### Bài 3 (điền TODO)

Điền chỗ trống của lớp (Cách 2):

In [29]:
class Ex3Queue:
    def __init__(self, k):
        self.q = [0] * k
        self.k = k
        self.front = 0
        self.size = 0

    def enQueue(self, value):
        if ...:                                   # TODO 1: khi nào đầy?
            return False
        self.q[...] = value                       # TODO 2: ô ghi (nhớ quay vòng)
        self.size += 1
        return True

    def Rear(self):
        if self.size == 0:
            return -1
        return self.q[...]                        # TODO 3: ô cuối hàng

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# x = Ex3Queue(3); x.enQueue(1); x.enQueue(2); assert x.Rear() == 2

### Bài 4 (tự viết lại)

Viết lại lớp **Cách 3 (hai con trỏ `head`/`tail`, mảng `k + 1` ô)** từ đầu, không nhìn bài giảng. Chú ý điều kiện đầy `(tail + 1) % n == head` và `Rear` lùi một ô.

In [30]:
class Ex4Queue:
    # TODO: tự viết đủ 7 phương thức: __init__, enQueue, deQueue, Front, Rear, isEmpty, isFull
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert run_ops(Ex4Queue, ex_ops, ex_args) == ex_expected

### Bài 5 (mini challenge, biến thể)

Viết lớp `MyCircularDeque(k)` (LeetCode 641) với các phương thức `insertFront`, `insertLast`, `deleteFront`, `deleteLast`, `getFront`, `getRear`, `isEmpty`, `isFull`. Gợi ý: giữ `front` + `size`; `insertFront` phải **lùi** `front`. Sau đó tự viết test ngẫu nhiên so với `collections.deque` có `maxlen` kiểm soát thủ công.

In [31]:
class MyCircularDeque:
    # TODO: tự viết lớp hàng đợi hai đầu vòng
    ...

# Khi đã viết xong, thử bằng test ở mục 19 (so với deque tham chiếu)

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): hàng đợi đầy khi số phần tử bằng sức chứa. Hai biến nào chứa hai giá trị đó?
- Mức 2 (vừa): ô ghi là ô "ngay sau phần tử cuối", tức `front` cộng với số phần tử, rồi quay vòng.
- Mức 3 (gần đáp án): `if self.size == self.k:`; `self.q[(self.front + self.size) % self.k] = value`; `return self.q[(self.front + self.size - 1) % self.k]`.

**Bài 4**

- Mức 1: mảng có `k + 1` ô; `head` là đầu, `tail` là ô sẽ ghi tiếp.
- Mức 2: rỗng `head == tail`; đầy `(tail + 1) % n == head`; `Rear` ở ô ngay trước `tail`.
- Mức 3: `__init__`: `self.n = k + 1; self.q = [0] * self.n; self.head = self.tail = 0`; `Rear`: `self.q[(self.tail - 1) % self.n]`.

**Bài 5**

- Mức 1: `insertLast` y hệt `enQueue`, `deleteFront` y hệt `deQueue`.
- Mức 2: `insertFront`: đầu hàng lùi một ô: `front = (front - 1) % k`, rồi ghi `q[front]`.
- Mức 3: `deleteLast` chỉ cần `size -= 1`; `getRear` = `q[(front + size - 1) % k]`.

## 19. Đáp án tham khảo

**Bài 1:** `k = 2`. Dòng 1: `enQueue(5)` True, `enQueue(6)` True, `enQueue(7)` False (đầy), `Front()` = `5`. In `True True False 5`. Dòng 2: `deQueue()` True (bỏ `5`), `enQueue(7)` True (ô 0), `Rear()` = `7`, `Front()` = `6`. In `True True 7 6`.

**Bài 2:** sau 4 lần `enQueue`: `q = [1,2,3,4]`, `front = 0`, `size = 4`. Hai `deQueue`: `front = 2`, `size = 2`. `enQueue(5)`: ô `(2+2)%4 = 0` → `q[0] = 5`; `enQueue(6)`: ô `(2+3)%4 = 1` → `q[1] = 6`. Kết quả `q = [5,6,3,4]`, `front = 2`, `size = 4`, `Front() = q[2] = 3`, `Rear() = q[(2+4-1)%4] = q[1] = 6`. Hàng đợi logic: `[3, 4, 5, 6]`.

In [32]:
# Bài 1
q = MyCircularQueue(2)
line1 = (q.enQueue(5), q.enQueue(6), q.enQueue(7), q.Front())
line2 = (q.deQueue(), q.enQueue(7), q.Rear(), q.Front())
assert line1 == (True, True, False, 5) and line2 == (True, True, 7, 6)
print("Bài 1:", *line1, "|", *line2)

# Bài 2
q = MyCircularQueue(4)
for v in (1, 2, 3, 4):
    q.enQueue(v)
q.deQueue(); q.deQueue()
q.enQueue(5); q.enQueue(6)
assert q.q == [5, 6, 3, 4] and q.front == 2 and q.size == 4
assert (q.Front(), q.Rear()) == (3, 6)
print("Bài 2:", q.q, q.front, q.size, q.Front(), q.Rear())

# Bài 3
class Ex3Answer:
    def __init__(self, k):
        self.q, self.k, self.front, self.size = [0] * k, k, 0, 0
    def enQueue(self, value):
        if self.size == self.k:
            return False
        self.q[(self.front + self.size) % self.k] = value
        self.size += 1
        return True
    def Rear(self):
        if self.size == 0:
            return -1
        return self.q[(self.front + self.size - 1) % self.k]

x = Ex3Answer(3); x.enQueue(1); x.enQueue(2)
assert x.Rear() == 2 and x.enQueue(3) and not x.enQueue(4) and x.Rear() == 3
print("Bài 3: OK")

# Bài 4
class Ex4Answer:
    def __init__(self, k):
        self.n = k + 1
        self.q = [0] * self.n
        self.head = self.tail = 0
    def enQueue(self, value):
        if self.isFull():
            return False
        self.q[self.tail] = value
        self.tail = (self.tail + 1) % self.n
        return True
    def deQueue(self):
        if self.isEmpty():
            return False
        self.head = (self.head + 1) % self.n
        return True
    def Front(self):
        return -1 if self.isEmpty() else self.q[self.head]
    def Rear(self):
        return -1 if self.isEmpty() else self.q[(self.tail - 1) % self.n]
    def isEmpty(self):
        return self.head == self.tail
    def isFull(self):
        return (self.tail + 1) % self.n == self.head

assert run_ops(Ex4Answer, ex_ops, ex_args) == ex_expected
rng = random.Random(4)
for _ in range(100):
    k = rng.randint(1, 6)
    ops, args = random_ops(rng, k, rng.randint(1, 100), 0.5)
    assert run_ops(Ex4Answer, ops, args) == oracle_run(ops, args)
print("Bài 4: OK (đối chiếu oracle)")

# Bài 5: kiểm tra CircularDeque (đã viết ở mục 15.1) với deque tham chiếu
rng = random.Random(5)
for _ in range(200):
    k = rng.randint(1, 6)
    dq, ref = CircularDeque(k), deque()
    for _ in range(100):
        act = rng.choice(["insertFront", "insertLast", "deleteFront", "deleteLast", "getFront", "getRear"])
        if act in ("insertFront", "insertLast"):
            v = rng.randint(0, 1000)
            got = getattr(dq, act)(v)
            if len(ref) < k:
                (ref.appendleft if act == "insertFront" else ref.append)(v)
                assert got is True
            else:
                assert got is False
        elif act in ("deleteFront", "deleteLast"):
            got = getattr(dq, act)()
            assert got == bool(ref)
            if ref:
                (ref.popleft if act == "deleteFront" else ref.pop)()
        elif act == "getFront":
            assert dq.getFront() == (ref[0] if ref else -1)
        else:
            assert dq.getRear() == (ref[-1] if ref else -1)
print("Bài 5: CircularDeque khớp deque tham chiếu")

Bài 1: True True False 5 | True True 7 6
Bài 2: [5, 6, 3, 4] 2 4 3 6
Bài 3: OK
Bài 4: OK (đối chiếu oracle)
Bài 5: CircularDeque khớp deque tham chiếu


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template hàng đợi vòng (mảng + `front` + `size`):**

```python
class MyCircularQueue:

    def __init__(self, k: int):
        self.q = [0] * k
        self.k = k
        self.front = 0
        self.size = 0

    def enQueue(self, value: int) -> bool:
        if self.size == self.k:
            return False
        self.q[(self.front + self.size) % self.k] = value
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.size == 0:
            return False
        self.front = (self.front + 1) % self.k
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.size == 0 else self.q[self.front]

    def Rear(self) -> int:
        if self.size == 0:
            return -1
        return self.q[(self.front + self.size - 1) % self.k]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.k
```

**Checklist trước khi submit:**

- [ ] Có `size` (hoặc mảng `k + 1` ô) để phân biệt rỗng và đầy?
- [ ] Mọi chỗ cộng chỉ số đều có `% k`?
- [ ] `Rear` dùng `front + size - 1` (không phải `front + size`)?
- [ ] `deQueue` vừa tiến `front` vừa giảm `size`; `enQueue` tăng `size`?
- [ ] `Front`/`Rear` trả `-1` khi rỗng; `enQueue`/`deQueue` trả `False` khi đầy/rỗng?
- [ ] Đã thử `k = 1`, hàng rỗng, hàng đầy, đầy-rồi-cạn-rồi-đầy, giá trị `0`?
- [ ] Mọi thao tác `O(1)`, Space `O(k)`?

**Ghi nhớ một câu:** *"Ghi ở (front + size) % k, đọc cuối ở (front + size - 1) % k, và dùng size để biết rỗng hay đầy."*